# 🌊 Geo-AI Training — Workshop 3: จำลองน้ำท่วม/ทิศทางการไหลของน้ำ

**แนวคิด**: เตรียม DEM ให้เหมาะกับงานวิเคราะห์น้ำ (ย่อขนาด + smooth + fill sink) แล้วคำนวณทิศทางการไหลของน้ำแบบ **D8** (แต่ละเซลล์ไหลไปยังเพื่อนบ้านที่ลาดชันมากที่สุด) สะสมปริมาณน้ำเพื่อดูเส้นทางน้ำไหล และจำลอง**สถานการณ์สมมติ**ว่าถ้าฝนตกเท่านี้ (มม.) จะท่วมตรงไหนบ้าง



## 🧩 Setup

> 📦 ก่อนรัน ติดตั้ง dependencies ให้ครบก่อน (ครั้งเดียวพอ):
> ```
> pip install -r requirements.txt
> ```

In [ ]:
# ติดตั้ง library ที่ใช้ในไฟล์นี้ (ใช้เวลาสักครู่ตอนรันครั้งแรก)
!pip install -q rasterio scikit-image scipy geopandas shapely folium gdown

print("ติดตั้งเสร็จแล้ว ✅")

### ติดตั้งฟอนต์ไทย (สำหรับกราฟที่มีข้อความไทย)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import urllib.request
import os

# โหลดฟอนต์ Sarabun (ฟอนต์ไทยจาก Google Fonts) มาใช้กับกราฟ — โหลดครั้งเดียว ถ้ามีไฟล์แล้วข้ามได้เลย
font_path = "Sarabun-Regular.ttf"
if not os.path.exists(font_path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/google/fonts/main/ofl/sarabun/Sarabun-Regular.ttf",
        font_path
    )
fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = "Sarabun"
plt.rcParams["axes.unicode_minus"] = False

print("ติดตั้งฟอนต์ไทยเสร็จแล้ว ✅")

In [ ]:
import os

# 📁 ใช้พื้นที่เก็บไฟล์ชั่วคราวในเครื่อง Colab (ไม่เชื่อม Google Drive)
# ⚠️ ไฟล์ในนี้จะหายไปเมื่อ Colab runtime ถูกตัดการเชื่อมต่อ/รีสตาร์ท — ดาวน์โหลดเก็บเองก่อนปิดเครื่อง
# (ใช้แผง Files ด้านซ้ายของ Colab คลิกขวาไฟล์ > Download)
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/outputs"
EXPORT_DIR = os.path.join(OUTPUT_DIR, "export")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("EXPORT_DIR :", EXPORT_DIR)

### โหลดข้อมูลจาก Google Drive

ดาวน์โหลดข้อมูลโดรนจริง 3 ไฟล์ (ถ้ามีอยู่แล้วใน `DATA_DIR` จะข้ามการโหลดซ้ำ)

In [ ]:
import gdown

FILES = {
    "orthophoto.tif": "1i3yprs-Es03CFbMKx2rTvcsXJs0KOX_r",
    "dtm.tif": "1FMfmeasnKm7XkqdH7OrBk-3ZCcXaNHzZ",
    "dsm.tif": "1tL881oAJjaDWaYKgADQYD9fjtm_incm0",
}

for filename, file_id in FILES.items():
    out_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(out_path):
        print(f"✅ มีไฟล์อยู่แล้ว: {filename}")
    else:
        print(f"⬇️  กำลังโหลด: {filename} ...")
        gdown.download(id=file_id, output=out_path, quiet=False)

print("\nเสร็จแล้ว พร้อมใช้งาน")

## 🔬 เตรียม DEM: ย่อขนาด + smooth + fill sink

> ⚠️ **หมายเหตุเรื่องความเร็ว**: DTM จริงมีขนาดหลายล้านพิกเซล ถ้าคำนวณทีละพิกเซลด้วย Python loop ตรงๆ จะช้ามาก (เป็นชั่วโมง) จึงลดขนาดภาพลงก่อน (downsample) ให้เหลือด้านละไม่เกิน ~300 พิกเซล ซึ่งเพียงพอสำหรับดูภาพรวมทิศทางน้ำไหลแล้ว

In [ ]:
import rasterio
import numpy as np
from rasterio.enums import Resampling
from rasterio.transform import Affine

TARGET_SIZE = 300  # 🔧 ปรับได้: ใหญ่ขึ้น = ละเอียดขึ้นแต่ช้าลงมาก (loop คำนวณทิศทางน้ำเป็น O(H*W))

with rasterio.open(os.path.join(DATA_DIR, "dtm.tif")) as src:
    orig_h, orig_w = src.height, src.width
    step = max(1, max(orig_h, orig_w) // TARGET_SIZE)
    H, W = orig_h // step, orig_w // step

    # resampling="average" = เฉลี่ยพิกเซลตอนย่อขนาด (ไม่ใช่สุ่มหยิบพิกเซลเดียวแบบ [::step])
    # ช่วยลดพื้นผิวขรุขระ/สิ่งแปลกปลอมที่เกิดจากการสุ่มตัวอย่างแบบเดิม
    dtm = src.read(1, out_shape=(H, W), resampling=Resampling.average).astype(np.float64)
    dtm_nodata = src.nodata
    dtm_transform = src.transform * Affine.scale(orig_w / W, orig_h / H)
    dtm_profile = src.profile.copy()
    dtm_profile.update(height=H, width=W, transform=dtm_transform)

if dtm_nodata is not None:
    dtm[dtm == dtm_nodata] = np.nan
if np.isnan(dtm).any():
    dtm = np.where(np.isnan(dtm), np.nanmean(dtm), dtm)

print(f"ขนาดต้นฉบับ: {orig_h} x {orig_w} พิกเซล")
print(f"ขนาดที่ใช้คำนวณ (เฉลี่ยทุกๆ {step} พิกเซล): {H} x {W} พิกเซล")

### Smooth ผิว DEM ด้วย Gaussian filter

พื้นผิวจาก photogrammetry มี noise ระดับพิกเซล (เช่น ยอดหญ้า/ใบไม้เล็กๆ) ทำให้เกิด "หลุมปลอม" ที่รบกวนการหาทิศทางน้ำไหล การ smooth เบาๆ ช่วยลด noise นี้ก่อนวิเคราะห์

In [ ]:
from scipy.ndimage import gaussian_filter

SMOOTH_SIGMA = 1.0  # 🔧 ปรับได้: ยิ่งมากยิ่งเรียบ แต่ถ้ามากไปจะเบลอรายละเอียดภูมิประเทศจริง

dtm_smooth = gaussian_filter(dtm, sigma=SMOOTH_SIGMA)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im0 = axes[0].imshow(dtm, cmap="terrain")
axes[0].set_title("ก่อน smooth")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(dtm_smooth, cmap="terrain")
axes[1].set_title(f"หลัง smooth (sigma={SMOOTH_SIGMA})")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
for a in axes:
    a.axis("off")
plt.tight_layout()
plt.show()

### Fill Sink (เติมหลุม/แอ่งที่ปิดตัน)

DEM มักมีหลุมเล็กๆ ที่ทำให้น้ำ "ไหลตกไปแล้วไปต่อไม่ได้" (D8 หาทางลงต่อไม่เจอ) การ fill sink คือยกระดับก้นหลุมขึ้นมาเท่าจุดที่น้ำจะล้นออกได้ ทำให้ทุกจุดมีทางไหลต่อเนื่องไปจนถึงขอบภาพ — ใช้เทคนิค morphological reconstruction (ไม่ต้องพึ่ง library เฉพาะทางอย่าง richdem/pysheds)

In [ ]:
from skimage.morphology import reconstruction

# seed = ค่าสูงสุดทุกจุด ยกเว้นขอบภาพ (คงค่าเดิมไว้เป็น "ทางออก") แล้วค่อยๆ "รด" น้ำลงมาจนกว่าจะติดผิว DEM จริง
seed = np.full_like(dtm_smooth, dtm_smooth.max())
seed[0, :] = dtm_smooth[0, :]
seed[-1, :] = dtm_smooth[-1, :]
seed[:, 0] = dtm_smooth[:, 0]
seed[:, -1] = dtm_smooth[:, -1]

dtm_filled = reconstruction(seed, dtm_smooth, method="erosion")
fill_depth = dtm_filled - dtm_smooth  # ความลึกของหลุมที่ถูกเติม (ใช้เป็น "ความจุแอ่ง" ตอนจำลองฝนตกด้วย)

# ⚠️ หลุมที่ถูกเติมจะกลายเป็น "พื้นที่ราบเรียบสนิท" (ทุกพิกเซลในหลุมมีค่าเท่ากันเป๊ะ)
# ถ้าเอา dtm_filled ไปหาทิศทางน้ำไหลตรงๆ D8 จะหาเพื่อนบ้านที่ "ต่ำกว่าจริง" ไม่เจอเลยสักจุดในพื้นที่ราบนี้
# (ทุกทิศชันเป็น 0 พอดี) ทำให้น้ำ "ค้าง" อยู่ในหลุมทุกจุด ไม่ไหลต่อไปตามทางออกจริง — สังเกตได้ว่านี่คือปัญหาเดิมที่ fill sink ควรจะแก้ แต่กลับเกิดซ้ำในรูปแบบใหม่
# วิธีแก้: บวกความชันเดิม (dtm_smooth) เข้าไปเล็กน้อยมากๆ (ไม่กระทบระดับชั้นของ fill sink) เพื่อ "เอียง" พื้นราบนั้นตามภูมิประเทศเดิมเบาๆ ให้มีทิศทางไหลชัดเจน
ROUTE_EPS = 1e-9  # 🔧 เล็กมากพอที่จะไม่กระทบระดับความสูงจริงระหว่างแอ่งต่างกัน
dtm_route = dtm_filled + ROUTE_EPS * dtm_smooth

print(f"จำนวนพิกเซลที่เป็นหลุม (ถูกเติม): {(fill_depth > 0.01).sum()} จาก {H*W} พิกเซล")
print(f"หลุมลึกที่สุดที่เติม: {fill_depth.max():.2f} เมตร")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im0 = axes[0].imshow(dtm_filled, cmap="terrain")
axes[0].set_title("DEM หลัง fill sink")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(fill_depth, cmap="Blues")
axes[1].set_title("ความลึกหลุมที่ถูกเติม (แอ่งธรรมชาติ)")
plt.colorbar(im1, ax=axes[1], fraction=0.046, label="เมตร")
for a in axes:
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ทิศทาง 8 ทิศรอบเซลล์ และระยะทางจริง (แนวทแยงไกลกว่าแนวตรง sqrt(2) เท่า)
NEIGHBORS = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
DIST = [np.sqrt(2),1,np.sqrt(2),1,1,np.sqrt(2),1,np.sqrt(2)]

# flow_to[y, x] = (ty, tx) คือเซลล์ปลายทางที่น้ำจากจุด (y,x) จะไหลไป
# ใช้ dtm_route (DEM ที่ fill sink + เอียงพื้นราบเล็กน้อยแล้ว) เพื่อให้ทุกจุดมีทางไหลต่อเนื่อง ไม่มีจุดที่ไหลตกแล้วค้าง
flow_to = np.full((H, W, 2), -1, dtype=np.int32)

print("กำลังคำนวณทิศทางน้ำไหล (ใช้เวลาสักครู่)...")
for y in range(H):
    for x in range(W):
        best_slope = 0.0
        best_target = None
        for (dy, dx), dist in zip(NEIGHBORS, DIST):
            ny, nx = y + dy, x + dx
            if 0 <= ny < H and 0 <= nx < W:
                slope = (dtm_route[y, x] - dtm_route[ny, nx]) / dist  # ความชัน = (สูง-ต่ำ)/ระยะทาง
                if slope > best_slope:
                    best_slope = slope
                    best_target = (ny, nx)
        if best_target is not None:
            flow_to[y, x] = best_target

print("เสร็จแล้ว ✅")

In [ ]:
# แสดงทิศทางน้ำไหลเป็นลูกศร (สุ่มแสดงห่างๆ กัน ไม่งั้นลูกศรจะทับกันจนอ่านไม่ออก)
ARROW_STEP = max(1, min(H, W) // 40)  # 🔧 ปรับได้: ยิ่งน้อยยิ่งมีลูกศรเยอะ/ถี่ขึ้น

ys_idx, xs_idx = np.meshgrid(np.arange(0, H, ARROW_STEP), np.arange(0, W, ARROW_STEP), indexing="ij")
dy = np.zeros_like(ys_idx, dtype=float)
dx = np.zeros_like(xs_idx, dtype=float)
for i in range(ys_idx.shape[0]):
    for j in range(ys_idx.shape[1]):
        y, x = ys_idx[i, j], xs_idx[i, j]
        ty, tx = flow_to[y, x]
        if ty >= 0:
            dy[i, j] = ty - y
            dx[i, j] = tx - x

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(dtm_filled, cmap="terrain")
ax.quiver(xs_idx, ys_idx, dx, -dy, color="red", scale=40, width=0.003)
ax.set_title("ทิศทางการไหลของน้ำ (ลูกศรแดง)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 🔬 สะสมปริมาณน้ำไหล (Flow Accumulation)

ไล่จากเซลล์สูงสุดไปต่ำสุด สะสมปริมาณน้ำไปยังเซลล์ปลายทางเรื่อยๆ

In [ ]:
acc = np.ones((H, W), dtype=np.float64)  # เริ่มต้นทุกเซลล์มีน้ำ 1 หน่วย

# ต้องเรียงตาม dtm_route (ตัวเดียวกับที่ใช้หาทิศทางน้ำไหล) ไม่ใช่ dtm_filled
# เพราะ dtm_route รับประกันว่าน้ำไหลจากค่าสูงไปต่ำอย่างเคร่งครัดเสมอ (ไม่มีที่ราบเป๊ะ) ลำดับนี้จึงถูกต้อง
order = np.argsort(-dtm_route.ravel())  # เรียงลำดับจากสูงไปต่ำ
ys_all, xs_all = np.unravel_index(order, dtm_route.shape)

for y, x in zip(ys_all, xs_all):
    ty, tx = flow_to[y, x]
    if ty >= 0:
        acc[ty, tx] += acc[y, x]

flow_acc = acc
log_flow_acc = np.log1p(flow_acc)  # ใช้ log ช่วยให้มองเห็นเส้นทางน้ำชัดขึ้น

print("คำนวณ flow accumulation เสร็จแล้ว ✅")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(log_flow_acc, cmap="Blues")
plt.colorbar(im, ax=ax, fraction=0.046, label="log(flow accumulation + 1)")
ax.set_title("ปริมาณน้ำไหลสะสม")
ax.axis("off")
plt.tight_layout()
plt.show()

### พื้นที่เสี่ยงน้ำท่วมโดยทั่วไป (จาก elevation + flow accumulation)

จุดต่ำ (25% ต่ำสุด) ที่มีน้ำไหลมารวมเยอะ (90 percentile บนสุดของ flow accumulation) — ยังไม่ผูกกับปริมาณฝนตกจริง เป็นแค่ "ความเสี่ยงเชิงภูมิประเทศ" ทั่วไป

In [ ]:
low_area = dtm_filled < np.percentile(dtm_filled, 25)
high_flow = flow_acc > np.percentile(flow_acc, 90)
flood_risk = low_area & high_flow

print(f"พื้นที่เสี่ยงน้ำท่วม: {flood_risk.sum()} พิกเซล ({flood_risk.mean()*100:.1f}% ของพื้นที่ทั้งหมด)")

### โหลด Orthophoto แล้ว reproject ให้ตรงกริดกับ DTM

เพื่อเอาไว้ซ้อนดูว่าพื้นที่เสี่ยงตรงกับอะไรจริงๆ บนภาพถ่าย (orthophoto เป็นคนละ CRS/กริดกับ DTM เหมือนใน Workshop 2 จึงต้อง reproject ก่อน)

In [ ]:
from rasterio.warp import reproject

with rasterio.open(os.path.join(DATA_DIR, "orthophoto.tif")) as src:
    ortho_src = src.read([1, 2, 3])
    ortho_src_crs = src.crs
    ortho_src_transform = src.transform

ortho_on_dtm = np.zeros((3, H, W), dtype=np.uint8)
reproject(
    source=ortho_src, destination=ortho_on_dtm,
    src_transform=ortho_src_transform, src_crs=ortho_src_crs,
    dst_transform=dtm_profile["transform"], dst_crs=dtm_profile["crs"],
    resampling=Resampling.average,
)
ortho_on_dtm_rgb = ortho_on_dtm.transpose(1, 2, 0)

print("Reproject orthophoto ให้ตรงกริดกับ DTM เสร็จแล้ว ✅")

In [ ]:
# สร้างภาพสีแดงทึบสำหรับพื้นที่เสี่ยง (ไม่ใช้ cmap ตรงๆ เพราะพิกเซลที่ไม่ถูก mask ออกมีค่าเดียวกันหมด
# ทำให้ matplotlib ปรับสเกลสีจนได้สีอ่อนเกือบขาวแทนที่จะเป็นสีแดงเข้ม)
risk_rgba = np.zeros((*flood_risk.shape, 4))
risk_rgba[flood_risk] = [1, 0, 0, 0.8]  # แดงทึบ (RGBA 0-1)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

axes[0].imshow(dtm_filled, cmap="terrain")
axes[0].imshow(risk_rgba)
axes[0].set_title("DTM + พื้นที่เสี่ยงน้ำท่วม")

axes[1].imshow(ortho_on_dtm_rgb)
axes[1].imshow(risk_rgba)
axes[1].set_title("Orthomosaic + พื้นที่เสี่ยงน้ำท่วม")

for a in axes:
    a.axis("off")
plt.tight_layout()
plt.show()

> 💡 **ต่อยอด**: บนพื้นที่จริงขนาดใหญ่ ลูป Python แบบนี้จะช้ามาก ควรลดขนาดภาพ (resample DTM ให้หยาบขึ้น) ก่อนคำนวณ หรือใช้ library ที่ optimize แล้ว

## 🌧️ จำลองสถานการณ์ฝนตก (สมมติ)

> ⚠️ **นี่คือการจำลองแบบง่ายเพื่อการศึกษาเท่านั้น ไม่ใช่แบบจำลองอุทกวิทยาจริง** สมมติฐาน:
> - **น้ำไม่ระบาย** — ไม่มีท่อระบายน้ำ ไม่มีการซึมลงดิน ไม่มีการระเหย
> - น้ำฝนที่ตกบนแต่ละเซลล์จะไหลไปสะสมตามเส้นทาง flow ที่คำนวณไว้ (ใช้ `flow_acc` ที่มีอยู่แล้ว)
> - แอ่ง/หลุมแต่ละจุดรับน้ำได้ไม่เกิน "ความจุ" ของมัน (`fill_depth` จากขั้นตอน fill sink ด้านบน) ถ้าน้ำมาเกินความจุ ส่วนเกินจะถือว่าล้นออกไป (ไม่ได้จำลองว่าล้นไปท่วมที่ไหนต่อ)
>
> แนวคิด: **ปริมาณน้ำที่มารวมที่จุดหนึ่ง = (จำนวนเซลล์ต้นน้ำที่ไหลมา) × (ปริมาณฝนต่อเซลล์)** แล้วเทียบกับความจุแอ่งที่จุดนั้น

In [ ]:
RAINFALL_MM = 100  # 🔧 ปรับได้: ปริมาณฝนสมมติ (มิลลิเมตร) — ลองเปลี่ยนดูว่าพื้นที่ท่วมเปลี่ยนไปแค่ไหน

rainfall_m = RAINFALL_MM / 1000  # แปลงหน่วยเป็นเมตร

# น้ำที่ไหลมารวมที่แต่ละเซลล์ ถ้าคิดเป็น "ความลึก" เทียบเท่า (ไม่คำนึงพื้นที่จริง เพราะ flow_acc นับเป็นจำนวนเซลล์อยู่แล้ว)
flood_depth_potential = flow_acc * rainfall_m

print(f"สมมติฝนตก {RAINFALL_MM} มม. ทั่วพื้นที่")
print(f"ความลึกน้ำที่อาจสะสมได้สูงสุด (ก่อนหักความจุแอ่ง): {flood_depth_potential.max():.2f} เมตร")

In [ ]:
# น้ำจะขังได้จริงเฉพาะจุดที่เป็นแอ่ง (fill_depth > 0) และไม่เกิน "ความจุ" ของแอ่งนั้น
flood_depth = np.minimum(flood_depth_potential, fill_depth)
flood_depth[fill_depth <= 0] = 0  # จุดที่ไม่ใช่แอ่ง (ไม่มีที่ขัง) น้ำจะไหลผ่านไปเรื่อยๆ ไม่นับว่าท่วมขัง

flood_extent = flood_depth > 0.01  # 🔧 ปรับได้: เกณฑ์ความลึกขั้นต่ำที่นับว่า "ท่วม" (เมตร)

n_flood_cells = flood_extent.sum()
print(f"จำนวนเซลล์ที่คาดว่าจะท่วมขัง: {n_flood_cells} จาก {H*W} เซลล์ ({flood_extent.mean()*100:.1f}%)")
if n_flood_cells > 0:
    print(f"ความลึกน้ำท่วมขังเฉลี่ย: {flood_depth[flood_extent].mean():.2f} เมตร, สูงสุด: {flood_depth.max():.2f} เมตร")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(dtm_filled, cmap="terrain", alpha=0.6)
im = ax.imshow(np.ma.masked_where(~flood_extent, flood_depth), cmap="Reds", vmin=0)
plt.colorbar(im, ax=ax, fraction=0.046, label="ความลึกน้ำท่วมขัง (เมตร)")
ax.set_title(f"จำลองน้ำท่วม: ฝนตก {RAINFALL_MM} มม. (สมมติ - น้ำไม่ระบาย)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 💾 Export ผลลัพธ์

In [ ]:
import geopandas as gpd

flow_profile = dtm_profile.copy()
flow_profile.update(dtype="float32")
with rasterio.open(os.path.join(EXPORT_DIR, "flow_accumulation.tif"), "w", **flow_profile) as dst:
    dst.write(flow_acc.astype(np.float32), 1)

print("✅ Export แล้ว: flow_accumulation.tif")

In [ ]:
from rasterio.features import shapes as rio_shapes
from shapely.geometry import shape as shp_shape

# พื้นที่เสี่ยงน้ำท่วมทั่วไป (จาก elevation + flow accumulation)
flood_risk_polys = [shp_shape(geom) for geom, val in
                     rio_shapes(flood_risk.astype(np.uint8), transform=dtm_profile["transform"]) if val == 1]
flood_risk_gdf = gpd.GeoDataFrame({"geometry": flood_risk_polys}, crs=dtm_profile["crs"])
if len(flood_risk_gdf):
    flood_risk_gdf.to_file(os.path.join(EXPORT_DIR, "flood_risk_zones.geojson"), driver="GeoJSON")

print("✅ Export แล้ว: flood_risk_zones.geojson")

In [ ]:
# ผลจำลองฝนตก (สมมติ) — แปลงเป็น vector พร้อมระบุ scenario ปริมาณฝนที่ใช้
sim_polys = [shp_shape(geom) for geom, val in
             rio_shapes(flood_extent.astype(np.uint8), transform=dtm_profile["transform"]) if val == 1]

flood_sim_gdf = gpd.GeoDataFrame({"geometry": sim_polys}, crs=dtm_profile["crs"])
flood_sim_gdf["rainfall_mm"] = RAINFALL_MM
if len(flood_sim_gdf):
    flood_sim_gdf.to_file(os.path.join(EXPORT_DIR, f"flood_simulation_{RAINFALL_MM}mm.geojson"), driver="GeoJSON")

print(f"✅ Export แล้ว: flood_simulation_{RAINFALL_MM}mm.geojson")

## 🌍 แผนที่ Interactive (Folium)

ดูผลลัพธ์ทั้งหมดซ้อนกันบนแผนที่ เลื่อน/ซูม/คลิกดูรายละเอียด และเปิด-ปิดแต่ละ layer ได้: DEM, ปริมาณน้ำไหลสะสม, พื้นที่เสี่ยงน้ำท่วมทั่วไป, และผลจำลองฝนตก

In [ ]:
import folium
from rasterio.transform import array_bounds
from rasterio.warp import transform_bounds as warp_transform_bounds

dtm_west, dtm_south, dtm_east, dtm_north = array_bounds(H, W, dtm_profile["transform"])
lon_min, lat_min, lon_max, lat_max = warp_transform_bounds(
    dtm_profile["crs"], "EPSG:4326", dtm_west, dtm_south, dtm_east, dtm_north)
image_bounds = [[lat_min, lon_min], [lat_max, lon_max]]

m = folium.Map(location=[(lat_min + lat_max) / 2, (lon_min + lon_max) / 2],
                zoom_start=17, tiles="CartoDB positron")
print("สร้างแผนที่ฐานเรียบร้อย ✅")

In [ ]:
# แปลง DEM และ flow accumulation เป็นภาพสีก่อนวางบนแผนที่
dtm_norm = (dtm_filled - dtm_filled.min()) / (dtm_filled.max() - dtm_filled.min())
dtm_rgb = (plt.get_cmap("terrain")(dtm_norm)[:, :, :3] * 255).astype(np.uint8)

flow_norm = log_flow_acc / max(log_flow_acc.max(), 1e-6)
flow_rgb = (plt.get_cmap("Blues")(flow_norm)[:, :, :3] * 255).astype(np.uint8)

folium.raster_layers.ImageOverlay(
    image=ortho_on_dtm_rgb, bounds=image_bounds, name="Orthomosaic", opacity=1.0,
).add_to(m)
folium.raster_layers.ImageOverlay(
    image=dtm_rgb, bounds=image_bounds, name="DEM", opacity=0.8, show=False,
).add_to(m)
folium.raster_layers.ImageOverlay(
    image=flow_rgb, bounds=image_bounds, name="Flow Accumulation", opacity=0.7, show=False,
).add_to(m)

print("เพิ่ม layer Orthomosaic, DEM และ Flow Accumulation แล้ว ✅")

In [ ]:
# พื้นที่เสี่ยงน้ำท่วมทั่วไป (vector, สีแดง)
if len(flood_risk_gdf):
    folium.GeoJson(
        flood_risk_gdf.to_crs(4326), name="พื้นที่เสี่ยงน้ำท่วม (ทั่วไป)",
        style_function=lambda f: {"fillColor": "red", "color": "red", "weight": 1, "fillOpacity": 0.6},
        show=False,
    ).add_to(m)

print("เพิ่ม layer พื้นที่เสี่ยงน้ำท่วมทั่วไปแล้ว ✅")

In [ ]:
# ผลจำลองฝนตก (vector, สีแดง พร้อม popup บอกปริมาณฝนที่จำลอง)
if len(flood_sim_gdf):
    folium.GeoJson(
        flood_sim_gdf.to_crs(4326), name=f"จำลองน้ำท่วม (ฝน {RAINFALL_MM} มม.)",
        style_function=lambda f: {"fillColor": "red", "color": "red", "weight": 1, "fillOpacity": 0.6},
        tooltip=folium.GeoJsonTooltip(fields=["rainfall_mm"], aliases=["ฝนตก (มม.):"]),
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
print("เพิ่ม layer ผลจำลองฝนตกแล้ว ✅")
m

---
✅ **จบ Workshop 3** — ไปต่อที่ `4_workshop4_traffic_cv.ipynb`